In [4]:
# ============================================================
# CrimePulse_7Nation — KPI Calculation (standalone, separate from EDA)
# Loads the 7 finalized CSVs, computes all 15 KPIs across 5 categories,
# prints them, and exports a kpi_summary.csv as ground-truth reference
# to cross-check against Tableau's calculated fields later.
#
# Save this file to:
# C:\Users\ABISHEK.000\OneDrive\Desktop\DA\Final\Crime_Records\Source_Code\Python
# ============================================================

import pandas as pd
import numpy as np
import os

FINAL_PATH = r"C:\Users\ABISHEK.000\OneDrive\Desktop\DA\Final\Crime_Records\Finalized"

# ------------------------------------------------------------
# LOAD FINALIZED TABLES (same files Tableau will connect to)
# ------------------------------------------------------------
crime_incident    = pd.read_csv(os.path.join(FINAL_PATH, "crime_incident_final.csv"))
victim_suspect    = pd.read_csv(os.path.join(FINAL_PATH, "victim_suspect_final.csv"))
case_resolution   = pd.read_csv(os.path.join(FINAL_PATH, "case_resolution_final.csv"))
financial_digital = pd.read_csv(os.path.join(FINAL_PATH, "financial_digital_final.csv"))

df = (
    crime_incident
    .merge(victim_suspect, on="crime_id", how="left")
    .merge(case_resolution, on="crime_id", how="left")
    .merge(financial_digital, on="crime_id", how="left")
)
assert len(df) == 950_000, "Row count mismatch after merge — check joins before trusting KPIs."

TOTAL_CRIMES = len(df)
kpi = {}  # flat dict for the summary export

print("=" * 60)
print("CRIME VOLUME & TREND KPIs")
print("=" * 60)

# 1. Total Crimes Reported + YoY % change
yearly_volume = df.groupby("incident_year").size()
yoy_change = yearly_volume.pct_change() * 100
kpi["total_crimes_reported"] = TOTAL_CRIMES
print(f"1. Total Crimes Reported: {TOTAL_CRIMES:,}")
print("   YoY % Change by Year:")
print(yoy_change.round(2).to_string())

# 2. Crime Volume Trend by Year
print("\n2. Crime Volume by Year:")
print(yearly_volume.to_string())

# 3. Crime Type Distribution (% share)
crime_type_share = (df["crime_type"].value_counts(normalize=True) * 100).round(2)
print("\n3. Crime Type Distribution (%):")
print(crime_type_share.to_string())

print("\n" + "=" * 60)
print("SEVERITY & RISK KPIs")
print("=" * 60)

# 4. Average Severity Score
avg_severity = df["severity_score"].mean()
kpi["avg_severity_score"] = round(avg_severity, 2)
print(f"4. Average Severity Score: {avg_severity:.2f}")

# 5. % Critical/High Severity Crimes
pct_critical_high = df["crime_severity"].isin(["Critical", "High"]).mean() * 100
kpi["pct_critical_high_severity"] = round(pct_critical_high, 2)
print(f"5. % Critical/High Severity: {pct_critical_high:.2f}%")

# 6. Severity Trend by Country
severity_by_country = df.groupby("country")["severity_score"].mean().round(2).sort_values(ascending=False)
print("\n6. Average Severity Score by Country:")
print(severity_by_country.to_string())

print("\n" + "=" * 60)
print("RESOLUTION & ENFORCEMENT KPIs")
print("=" * 60)

# 7. Case Resolution Rate (%) — denominator = total incidents, per earlier decision
resolution_rate = df["crime_status"].isin(["Solved", "Closed"]).mean() * 100
kpi["resolution_rate_pct"] = round(resolution_rate, 2)
print(f"7. Case Resolution Rate: {resolution_rate:.2f}%")

# 8. Arrest Rate (%)
arrest_rate = (df["arrested_flag"] == "Yes").mean() * 100
kpi["arrest_rate_pct"] = round(arrest_rate, 2)
print(f"8. Arrest Rate: {arrest_rate:.2f}%")

# 9. Average Case Duration (days) — excludes open/null cases by definition
avg_duration = df["case_duration_days"].mean()
kpi["avg_case_duration_days"] = round(avg_duration, 2)
open_cases_pct = df["case_duration_days"].isna().mean() * 100
kpi["pct_open_cases"] = round(open_cases_pct, 2)
print(f"9. Average Case Duration: {avg_duration:.2f} days  (excludes {open_cases_pct:.2f}% open/unresolved cases)")

print("\n" + "=" * 60)
print("FINANCIAL IMPACT KPIs")
print("=" * 60)

# 10. Total Financial Loss (USD)
total_loss = df["financial_loss_usd"].sum()
kpi["total_financial_loss_usd"] = round(total_loss, 2)
print(f"10. Total Financial Loss: ${total_loss:,.2f}")

# 11. Average Financial Loss per Financial Crime
financial_types = ["Embezzlement", "Currency Counterfeiting", "Capital Flight",
                    "Corporate Espionage", "Cyber Warfare", "Ransomware Extortion"]
avg_loss_financial = df.loc[df["crime_type"].isin(financial_types), "financial_loss_usd"].mean()
kpi["avg_loss_per_financial_crime"] = round(avg_loss_financial, 2)
print(f"11. Avg Financial Loss per Financial Crime: ${avg_loss_financial:,.2f}")

# 12. Top 5 Countries by Financial Loss
top5_countries_loss = df.groupby("country")["financial_loss_usd"].sum().sort_values(ascending=False).head(5)
print("\n12. Top 5 Countries by Financial Loss:")
print(top5_countries_loss.round(2).to_string())

print("\n" + "=" * 60)
print("GEOSPATIAL & CROSS-BORDER KPIs")
print("=" * 60)

# 13. Crime Density by Country/City (top 10 cities)
top10_cities = df.groupby(["country", "city"]).size().sort_values(ascending=False).head(10)
print("13. Top 10 Cities by Crime Density:")
print(top10_cities.to_string())

# 14. % Cross-Border Crimes
pct_cross_border = (df["cross_border_flag"] == "Yes").mean() * 100
kpi["pct_cross_border_crimes"] = round(pct_cross_border, 2)
print(f"\n14. % Cross-Border Crimes: {pct_cross_border:.2f}%")

# 15. % Digital Crimes
pct_digital = (df["digital_crime_flag"] == "Yes").mean() * 100
kpi["pct_digital_crimes"] = round(pct_digital, 2)
print(f"15. % Digital Crimes: {pct_digital:.2f}%")

print(f"\nKPI Calculation Is Completed")


CRIME VOLUME & TREND KPIs
1. Total Crimes Reported: 950,000
   YoY % Change by Year:
incident_year
2010      NaN
2011     6.78
2012     4.73
2013     6.58
2014    15.60
2015    10.13
2016     7.59
2017     8.24
2018    13.56
2019    12.29
2020    -5.73
2021   -23.08
2022     7.13
2023     7.66
2024   -10.86
2025    -3.25
2026   -57.43

2. Crime Volume by Year:
incident_year
2010    36087
2011    38534
2012    40358
2013    43013
2014    49725
2015    54764
2016    58923
2017    63776
2018    72424
2019    81326
2020    76665
2021    58971
2022    63174
2023    68010
2024    60625
2025    58655
2026    24970

3. Crime Type Distribution (%):
crime_type
Corporate Espionage        12.70
Drug Trafficking           12.44
Cyber Warfare              11.79
Ransomware Extortion       10.98
Embezzlement                9.83
Currency Counterfeiting     9.53
Armed Robbery               8.90
Capital Flight              8.69
Intentional Homicide        5.36
Forcible Abduction          5.27
Manslaughte